# LC #347 — Top K Frequent Elements
**Category:** HashMap + Heap (or Bucket Sort) | **Difficulty:** Medium | **Day 1**

---

<div style="padding:15px;border-left:8px solid #667eea;background:#f0f0ff;border-radius:4px;">
<strong>The Problem:</strong> Given an integer array <code>nums</code> and integer <code>k</code>,
return the <code>k</code> most frequent elements. Answer may be in any order.
</div>

**Examples:**
```
nums = [1, 1, 1, 2, 2, 3],  k = 2  → [1, 2]
nums = [1],                  k = 1  → [1]
```

**Core Insight:** Always decompose Top-K into two steps: (1) count frequencies with a hashmap, (2) find the top k by frequency. Step 2 can be done with a min-heap in O(n log k) or bucket sort in O(n) — know both.

## Mental Models

**1. Count First, Rank Second**
Never try to do both in one pass. Build the frequency map. Then find the top k from it. This clean separation makes both problems simple.

**2. Min-Heap of Size K**
Maintain a min-heap with exactly k elements (by frequency). For each new element: if its frequency > the min in the heap, pop the min and push the new one. At the end, the heap contains the top k. O(n log k) — better than O(n log n) sort when k << n.

**3. Bucket Sort Insight (O(n))**
Frequency is bounded: it can't exceed `len(nums)`. Use frequency as an array index: `buckets[freq].append(num)`. Iterate buckets from high to low, collecting elements until you have k. No comparisons needed — O(n) total.

In [ ]:
# Approach 1 — HashMap + Full Sort  O(n log n)
# Count frequencies, sort all items by frequency, take top k.

from collections import Counter

def topKFrequent_sort(nums, k):
    count = Counter(nums)
    # sort by frequency descending, take first k keys
    return sorted(count, key=count.get, reverse=True)[:k]

print(topKFrequent_sort([1, 1, 1, 2, 2, 3], 2))   # [1, 2]

# Approach 2 — heapq.nlargest  (O(n log k), concise)
import heapq

def topKFrequent_nlargest(nums, k):
    count = Counter(nums)
    return heapq.nlargest(k, count.keys(), key=count.get)

print(topKFrequent_nlargest([1, 1, 1, 2, 2, 3], 2))  # [1, 2]

## Trace: Bucket Sort Approach

Input: `nums = [1, 1, 1, 2, 2, 3]`, `k = 2`

**Step 1 — Frequency count:**
```
Counter: {1: 3, 2: 2, 3: 1}
```

**Step 2 — Build frequency buckets:**
```
len(nums) = 6, so buckets has indices 0..6
buckets[3] = [1]    (element 1 appears 3 times)
buckets[2] = [2]    (element 2 appears 2 times)
buckets[1] = [3]    (element 3 appears 1 time)
```

**Step 3 — Collect from highest bucket down:**
```
i=6: [] — skip
i=5: [] — skip
i=4: [] — skip
i=3: [1] → result = [1]       (len=1, need 2)
i=2: [2] → result = [1, 2]    (len=2, done!)
Return [1, 2] ✓
```

In [ ]:
# Optimal — Bucket Sort, O(n) time, O(n) space
# Frequency bounded by len(nums) — use as array index.

from collections import Counter

def topKFrequent(nums: list[int], k: int) -> list[int]:
    count = Counter(nums)

    # buckets[freq] = list of numbers with that frequency
    # max possible frequency = len(nums)
    buckets = [[] for _ in range(len(nums) + 1)]
    for num, freq in count.items():
        buckets[freq].append(num)

    # Collect from highest frequency down until we have k elements
    result = []
    for freq in range(len(buckets) - 1, 0, -1):
        result.extend(buckets[freq])
        if len(result) >= k:
            return result[:k]

    return result  # shouldn't reach here if k <= len(unique elements)

# Test
print(topKFrequent([1, 1, 1, 2, 2, 3], 2))    # [1, 2]
print(topKFrequent([1], 1))                    # [1]
print(topKFrequent([4, 1, -1, 2, -1, 2, 3], 2))  # [-1, 2]

## Complexity Analysis

| Approach | Time | Space | Notes |
|----------|------|-------|-------|
| HashMap + full sort | O(n log n) | O(n) | Overkill — sorting all unique elements |
| `heapq.nlargest(k, ...)` | O(n log k) | O(n + k) | Good when k << n |
| **Bucket sort** | **O(n)** | **O(n)** | **Bounded frequency = linear** |

**Why heap is O(n log k) not O(n log n):**
A min-heap of fixed size k: each push/pop is O(log k). We do at most n heap operations. Total: O(n log k). When k = 1, this is O(n). When k = n, it's O(n log n) — same as sort.

**Bucket sort intuition:** Counting sort is O(n + range). Here range = n (max frequency ≤ array length), so O(n + n) = O(n). The key is that the range is bounded.

## Interview Q&A

**Q: When would you use heap over bucket sort in production?**
A: When memory matters — heap uses O(k) extra space vs O(n) for buckets. When k is very small (top 3 of 1 million), heap is the right call. When k is large or you need O(n), use buckets.

**Q: What is `Counter` under the hood?**
A: A subclass of `dict`. `Counter(nums)` iterates once, incrementing counts. Time: O(n). Access pattern: `count[key]` = 0 for missing keys (doesn't raise KeyError).

**Q: Why does `heapq.nlargest` outperform a full sort?**
A: `nlargest(k, ...)` uses a min-heap of size k. It processes all n elements in O(n log k) instead of sorting all n in O(n log n). Python's implementation is especially efficient for small k — it switches to a simple scan when k=1.

**Q: How is Top-K relevant in data engineering?**
A: Everywhere. Top-K slow queries (query optimization). Top-K error codes (alert prioritization). Top-K resource consumers (capacity planning). At Citi: which 10 servers out of 6,000 are consistently the highest CPU consumers? Exactly this pattern.

**Q: What if there are ties at position k?**
A: The problem says any valid answer is acceptable. In practice for Top-K problems in production, you'd sort by secondary key (name, ID) to make results deterministic.

## The Citi Angle

**Top-K is a capacity planning primitive.** At Citi, out of 6,000 monitored endpoints:
- Which 10 servers have the highest average CPU over 30 days? → Top-K by average.
- Which 5 servers trigger the most alerts? → Top-K by alert count.
- Which application tier consumes the most memory? → Top-K by aggregate consumption.

**Direct code analogy:**
```python
# Top 10 servers by average CPU (last 30 days)
from collections import Counter
# cpu_readings: {server_id: [list of cpu_pct values]}
avg_cpu = {sid: sum(vals)/len(vals) for sid, vals in cpu_readings.items()}

# heapq approach for large fleet:
import heapq
top_10 = heapq.nlargest(10, avg_cpu, key=avg_cpu.get)
```

**Interview tie-in:** "Top-K queries are one of the most common patterns in monitoring and observability. At Citi, weekly capacity reports always started with: 'Show me the top 10 most stressed servers.' That's exactly this problem — count, then rank."

## Summary

| | Value |
|--|--|
| **Pattern** | Count frequencies (HashMap), then find top K |
| **Heap approach** | O(n log k) time, O(n + k) space — use when k << n |
| **Bucket approach** | O(n) time, O(n) space — use when you need true linear |
| **Key insight** | Frequency ≤ len(nums) → use frequency as array index |
| **Say in interview** | "Two-step: count with Counter, then heap of size k. Or bucket sort for O(n)." |

**Two templates:**

```python
# Template A: Heap (O(n log k))
count = Counter(nums)
return heapq.nlargest(k, count, key=count.get)

# Template B: Bucket Sort (O(n))
count = Counter(nums)
buckets = [[] for _ in range(len(nums) + 1)]
for num, freq in count.items(): buckets[freq].append(num)
result = []
for freq in range(len(buckets)-1, 0, -1):
    result.extend(buckets[freq])
    if len(result) >= k: return result[:k]
```